# TTS bootcamp on Kaggle: Task 3 (English words in Devanagari) audio + scoring

Before running: **Settings → Accelerator = GPU T4**, **Internet = On** (needs phone-verified account),
and **Add-ons → Secrets → HF_TOKEN** (your Hugging Face token, with the IndicF5 terms accepted).
Attach the dataset made from `tts_kit.zip` (Add Input). Then **Run All**. Every heavy step runs in its own
subprocess, so no kernel restart is needed.

In [ ]:
import os
os.environ["HF_TOKEN"] = "__HF_TOKEN__"

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path

src = next(p.parent for p in Path("/kaggle/input").rglob("metrics.py"))
KIT = Path("/kaggle/working/tts_kit")
if not KIT.exists():
    shutil.copytree(src, KIT)
os.chdir(KIT)
print("kit:", KIT, "->", sorted(os.listdir(KIT)))

try:
    if os.environ.get("HF_TOKEN", "").startswith("hf_"):
        print("HF_TOKEN already set in this session")
    else:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("Could not read the HF_TOKEN secret (" + type(e).__name__ + "). Add it under Add-ons > Secrets and tick it,")
    print("or paste the token below for this session only (it is not saved anywhere).")
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ").strip()
assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN missing or malformed"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
Path("out").mkdir(exist_ok=True)
t3 = next(p.parent for p in Path("/kaggle/input").rglob("en2dev.py"))
(KIT / "code").mkdir(exist_ok=True)
shutil.copy(t3 / "en2dev.py", KIT / "code" / "en2dev.py")
shutil.copy(t3 / "t3_idea_text.csv", KIT / "predictions" / "t3_idea_text.csv")
for f in ["heldout_L.csv", "heldout_D.csv"]:
    shutil.copy(t3 / f, KIT / "data" / f)
print("Task 3 files copied from", t3)

def sh(cmd, retry_on=None, show=12000):
    """Run a command in a fresh process; if it asks for the one-time length-fix restart, simply run it again."""
    print("$", cmd[:200])
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode and retry_on and retry_on in out:
        print("(length fix was just applied: running again, this happens once)")
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        out = (r.stdout or "") + (r.stderr or "")
    print(out[-show:])
    if r.returncode:
        raise RuntimeError(f"command failed (exit {r.returncode}): {cmd[:120]}")
    return out

In [ ]:
print(sh("nvidia-smi -L"))
print(sh("python -c \"import torch,sys;print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())\""))

## 1. Install (numpy pinned as the kit says; IndicF5 from GitHub main with --no-deps because its setup.py wants numpy<=1.26)

In [ ]:
sh("pip install -q 'numpy>=2.1,<2.3' 'scipy>=1.15,<2' 'transformers==4.49.0' accelerate soundfile librosa pyyaml pandas 'speechbrain>=1.0' huggingface_hub")
sh("pip install -q --no-deps git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7")
sh("pip install -q cached_path click datasets ema_pytorch hydra-core jieba matplotlib pydub pypinyin safetensors tomli torchdiffeq tqdm transformers_stream_generator vocos x_transformers")
sh("python -c \"import numpy,transformers;print('numpy',numpy.__version__,'transformers',transformers.__version__)\"")

## 2. Task 3 on validation: Task 2 text + English words in Devanagari -> `val_idea/` (same seed 0)

In [ ]:
sh("python baseline/run_baseline.py --items data/val_items.csv --out predictions/val_idea "
   "--text-csv predictions/t3_idea_text.csv --text-column idea_text --seed 0", retry_on="length fix")
out = sh("python metrics.py predictions/val_idea --json out/val_idea.json --csv out/val_idea_scores.csv")
Path("out/metrics_val_idea.log").write_text(out, encoding="utf-8")

## 3. Task 3 on the 40 evaluation items -> `t3_idea/`

In [ ]:
sh("python baseline/run_baseline.py --items data/eval_items.csv --out predictions/t3_idea "
   "--text-csv predictions/t3_idea_text.csv --text-column idea_text --seed 0", retry_on="length fix")
print(len(list(Path("predictions/t3_idea").glob("*.wav"))), "wav files in predictions/t3_idea")

## 4. Extra check (not a deliverable): 12 new sentences with new English words, Latin vs Devanagari text

In [ ]:
for tag in ["L", "D"]:
    sh(f"python baseline/run_baseline.py --items data/heldout_{tag}.csv --out predictions/heldout_{tag} --seed 0", retry_on="length fix")
    out = sh(f"python metrics.py predictions/heldout_{tag} --items data/heldout_{tag}.csv --json out/heldout_{tag}.json --csv out/heldout_{tag}_scores.csv")
    Path(f"out/metrics_heldout_{tag}.log").write_text(out, encoding="utf-8")

## 5. Bundle everything to download

In [ ]:
import zipfile
zp = Path("/kaggle/working/tts_outputs.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["predictions/t3_idea/*.wav", "predictions/val_idea/*.wav", "predictions/heldout_*/*.wav", "out/*"]:
        for p in Path(".").glob(pat):
            if p.is_file():
                z.write(p, p.as_posix())
print(zp, round(zp.stat().st_size / 1e6, 1), "MB")
print(json.dumps(json.load(open("out/val_idea.json")), ensure_ascii=False, indent=1))